# Claude Code 팀 교육 자료

> 이 자료는 Claude Code를 처음 사용하는 팀원들을 위한 실전 가이드입니다.  

> 설치 : Node.js 필요  
    - > npm install -g @anthropic-ai/claude-code

---

## 1. Claude Code는 어떻게 동작하는가

### 에이전트 루프

Claude Code는 단순한 자동완성이 아니라 **에이전트**입니다. 요청을 받으면 아래 루프를 반복합니다.

```
프롬프트 입력
  → 컨텍스트 수집 (파일, git 상태, CLAUDE.md, 자동 메모리 등)
  → 행동 실행 (파일 수정, 명령어 실행, 검색 등)
  → 결과 검증
  → 완료 또는 루프 반복
```

Claude가 접근할 수 있는 것들:
- 프로젝트 파일 및 디렉토리
- 터미널 명령 실행 결과
- git 상태 및 변경 이력
- `CLAUDE.md` — 프로젝트 지침 파일 (아래에서 자세히)
- 자동 메모리 — Claude가 세션마다 학습한 패턴
- 확장 연결 — MCP 서버, 브라우저(Claude in Chrome), Subagents 등

### 실수했을 때 되돌리기

모든 파일 편집은 되돌릴 수 있습니다.

- `Esc` 두 번 → 이전 상태로 즉시 복구
- Claude에게 직접 "방금 작업 취소해줘" 요청도 가능

---

## 2. CLAUDE.md — Claude의 행동 기준

### CLAUDE.md란?

프로젝트, 개인 워크플로우, 또는 팀 전체에 걸쳐 Claude에게 **지속적인 지침을 제공하는 마크다운 파일**입니다.  
세션이 시작될 때마다 자동으로 로드되며, Claude의 행동 기준이 됩니다.

> 주의: CLAUDE.md는 시스템 프롬프트가 아니라 **사용자 메시지**로 전달됩니다.  
> 덕분에 프롬프트 캐싱으로 토큰을 절약할 수 있지만, 시스템 프롬프트만큼 엄격하게 지켜지지 않을 수 있습니다.

저장 위치:
```
./CLAUDE.md          # 또는
./.claude/CLAUDE.md
```

### CLAUDE.md vs 자동 메모리

|  | CLAUDE.md | 자동 메모리 |
|---|---|---|
| **작성자** | 사람 | Claude (자동) |
| **포함 내용** | 지침, 규칙, 표준 | 학습된 패턴, 선호도 |
| **범위** | 프로젝트 / 사용자 / 조직 | 작업 트리 단위 |
| **로드 방식** | 모든 세션 전체 로드 | 모든 세션 첫 200줄 |
| **활용 예** | "TypeScript 사용, ESLint 적용" | "이 프로젝트는 yarn build 사용" |

→ CLAUDE.md는 **내가 Claude에게 주는 지침**, 자동 메모리는 **Claude가 스스로 학습한 내용**입니다.

### CLAUDE.md 처음 만들기

```bash
/init
```

Claude가 프로젝트를 분석해서 초안을 만들어 줍니다.  
이후 불필요한 항목을 지우고 필요한 항목을 추가하세요.

### 잘 쓰는 4가지 원칙

**① 크기: 200줄 이하**  
파일이 길수록 Claude의 준수율이 떨어집니다. 컨텍스트(토큰)도 더 소비됩니다.  
200줄을 넘으면 `@import` 또는 `.claude/rules/` 파일로 분리하세요.

**② 구조: 마크다운 헤더와 목록으로 정리**  
```markdown
## 코딩 컨벤션
- 함수는 단일 책임 원칙을 따를 것
- 변수명은 camelCase, 상수는 UPPER_SNAKE_CASE

## 금지 사항
- `any` 타입 사용 금지
- console.log 대신 logger 모듈 사용
```

**③ 구체성: 애매한 표현 배제**  
❌ "좋은 코드를 작성할 것"  
✅ "함수 하나는 하나의 역할만 수행할 것. 50줄을 초과하면 분리할 것"

**④ 일관성: 규칙끼리 충돌하지 않도록**  
예를 들어 "에러는 throw로 처리"와 "에러는 Result 패턴으로 처리"가 동시에 있으면 Claude가 혼란스러워합니다.

### 외부 파일 가져오기 (`@import`)

CLAUDE.md가 커지면 `@` 구문으로 분리된 파일을 불러올 수 있습니다.

```markdown
## 참고 자료
- 프로젝트 개요: @README.md
- 사용 가능한 명령어: @package.json

## 세부 지침
- git 워크플로우: @docs/git-instructions.md
- 개인 설정 재정의: @~/.claude/my-project-instructions.md
```

### 규칙 모듈화 (`.claude/rules/`)

파일 경로별로 다른 규칙을 적용하고 싶다면 `.claude/rules/` 디렉토리를 활용합니다.

```
your-project/
├── CLAUDE.md                    ← 목차 역할 (짧고 간결하게)
└── .claude/
    └── rules/
        ├── code-style.md        ← 코딩 스타일
        ├── testing.md           ← 테스트 규칙
        └── security.md          ← 보안 요구사항
```

각 파일에는 해당 영역에 맞는 지침을 작성하고, CLAUDE.md에서 `@` 구문으로 참조합니다.

```markdown
## 세부 지침
- 코딩 스타일: @.claude/rules/code-style.md
- 테스트 규칙: @.claude/rules/testing.md
- 보안 요구사항: @.claude/rules/security.md
```

---

## 3. 컨텍스트 관리 — 성능을 유지하는 핵심

### 컨텍스트 부패(Context Rot)란?

대화가 길어질수록 Claude의 성능이 저하되는 현상입니다.  
컨텍스트 사용률이 90% 이상이 되면:

- 버그 발생 빈도 증가
- 아키텍처 결정의 일관성 저하
- 프로젝트 고유 패턴을 망각

→ **적절한 컨텍스트 활용률을 유지하면 코드 양은 줄더라도 품질이 높아집니다.**

### 컨텍스트 관리 명령어

| 명령어 | 역할 |
|--------|------|
| `/context` | 현재 컨텍스트 사용량 확인 |
| `/compact [지침]` | 대화 압축. 핵심만 남기고 토큰 절약 |
| `/clear` | 대화 이력 완전 초기화 |
| `/mcp` | MCP 서버별 토큰 사용량 확인 |

사용 예:
```bash
# API 변경사항에 집중하여 압축
/compact focus on the API changes

# 작업 전환 전 초기화
/clear
```

### 세션 관리

```bash
claude --continue          # 이전 세션 이어서 시작
claude --resume            # 세션 목록에서 선택해 재개

claude -n feature-login    # 세션에 이름 지정
claude --resume feature-login  # 이름으로 재개

# 특정 시점에서 분기 (기존 이력 유지하며 새 세션 생성)
claude --continue --fork-session
```

### Skills vs Subagents

**Skills**는 반복되는 작업 절차를 문서화해둔 것입니다. 필요할 때 주 세션에 로드해서 Claude가 참고하며 실행합니다.

**Subagents**는 작업 자체를 별도의 독립적인 에이전트에게 **위임**하는 방식입니다. Claude Code 자체가 "요청 → 툴 실행 → 검증"을 반복하는 에이전트인데, Subagent는 그 안에서 특정 작업을 분리해 실행하는 하위 에이전트입니다. 파일 수정, 명령 실행 등 여러 단계에 걸친 작업을 주 세션과 완전히 분리된 컨텍스트에서 처리하고, 완료되면 결과만 반환합니다.

| | Skills | Subagents |
|---|---|---|
| **개념** | 절차 문서 (참고용) | 독립 실행 에이전트 (위임) |
| **실행 위치** | 주 세션 안에서 | 별도 컨텍스트에서 |
| **적합한 용도** | 반복 작업 절차, 워크플로우 가이드 | 독립적으로 완결되는 큰 작업 |

→ Subagent는 "토큰을 안 쓰는 게" 아니라, 주 세션의 컨텍스트 윈도우를 차지하지 않는다는 의미입니다.

---

## 4. 효율적인 워크플로우

### PlanMode (계획 모드)

코드를 바로 수정하기 전에, Claude가 **읽기 전용**으로 코드베이스를 분석하고 계획을 먼저 세우게 합니다.

활성화 방법:
```bash
Shift + Tab                       # 토글 (가장 빠름)
claude --permission-mode plan     # CLI 옵션
```

언제 쓰나요?

- **다단계 구현**: 여러 파일에 걸친 기능을 추가할 때
- **코드 탐색**: 낯선 코드베이스를 처음 파악할 때
- **방향 정렬**: 구현 전에 Claude와 접근 방식을 먼저 맞추고 싶을 때

실전 예:

> ❌ 바로 실행: "사용자 인증 기능 추가해줘"  
> ✅ PlanMode 활성화 → "사용자 인증 구현 계획 세워줘" → 계획 검토 → 승인 후 구현

### 3단계 작업 분리 (Context Isolation)

컨텍스트 부패를 막기 위한 실전 루틴입니다.

```
1단계: 작업을 독립적인 단계로 나누기
2단계: 각 단계 완료 후 테스트 & git commit
3단계: /clear 또는 새 세션으로 컨텍스트 초기화 후 다음 단계 시작
```

예시 — 로그인 기능 구현:
```bash
# 1단계: 백엔드 API
"POST /auth/login 엔드포인트 구현해줘"
→ 완료 → 테스트 → git commit → /clear

# 2단계: 토큰 로직
"JWT 발급 및 검증 로직 추가해줘"
→ 완료 → 테스트 → git commit → /clear

# 3단계: 프론트엔드 연동
"로그인 폼과 API 연동해줘"
```

> 💡 프론트엔드 ↔ 백엔드처럼 성격이 다른 작업으로 전환할 때는 반드시 세션을 새로 시작하세요.  
> 대화 이력이 쌓이면 컨텍스트가 비대해지고 현재 작업과 무관한 토큰이 낭비됩니다.

### 프롬프트 잘 쓰는 법

**간결하고 직접적으로 작성하세요.**

| 비교 | 예시 |
|------|------|
| ❌ 배경 설명 과다 | "저번에 말씀드렸던 것처럼 이 프로젝트는 Node.js 기반이고 TypeScript를 쓰는데요, 팀에서 ESLint도 쓰고 있어서..." |
| ✅ 목표 우선 명시 | "기존 auth 미들웨어에 JWT 만료 처리 로직 추가. 만료 시 401 반환, 에러 메시지는 'Token expired'" |

**예제는 말보다 강합니다 (Few-Shot).**  
원하는 결과물의 형식이나 스타일이 있다면, 설명하지 말고 예시를 직접 보여주세요.

```
❌ "함수 주석을 잘 달아줘"
✅ "함수 주석을 아래 형식으로 달아줘:
   /**
    * 사용자 인증 토큰을 검증합니다.
    * @param token - Bearer 토큰 문자열
    * @returns 검증된 사용자 ID 또는 null
    */
"
```

---

## 5. 하네스 엔지니어링 — 팀 수준의 안정적인 운용

### 하네스(Harness)란?

AI 에이전트가 **궤도를 이탈하지 않도록 감싸는 피드백 루프와 가이드레일 환경 전체**를 뜻합니다.  
개인 사용을 넘어 팀 수준에서 Claude Code를 안정적으로 운용하려면 하네스가 필요합니다.

> OpenAI 실험에서도 "하나의 큰 지침 파일" 방식은 실패했습니다.  
> 중요한 제약 조건을 Claude가 놓치거나, 모든 것이 중요하다 보니 오히려 무시하는 현상이 발생했습니다.  
> 해결책: **CLAUDE.md를 백과사전이 아닌 목차(Map)로 운용**

### 하네스 구성 요소

#### ① CLAUDE.md는 목차(Map)로

지침 파일이 너무 크면 Claude는 패턴 매칭만 수행하고 실제 규칙은 무시합니다.  
CLAUDE.md는 짧게 유지하고, 세부 내용은 분리된 규칙 파일로 링크하세요.

```markdown
# CLAUDE.md (목차형)
이 프로젝트는 Node.js + TypeScript 기반 API 서버입니다.

## 세부 지침
- 코딩 스타일: @.claude/rules/code-style.md
- 테스트 규칙: @.claude/rules/testing.md
- 보안 요구사항: @.claude/rules/security.md
```

#### ② 에이전트 실수 → 지침 누적

하네스의 기본 원리: **Claude가 실수할 때마다 같은 실수를 방지하는 지침을 CLAUDE.md에 추가합니다.**

```
Claude가 잘못된 패턴 사용
  → 수동 수정
  → CLAUDE.md에 해당 규칙 추가
  → 다음 세션부터 자동 방지
```

#### ③ 린터(Linter) + CI 연동

Claude가 생성한 코드가 기존 아키텍처 규칙을 따르는지 **자동 검증 장치**를 연결합니다.

```
Claude 코드 생성
  → 린터 / CI 실행
  → 실패 시: 에러 메시지 + 수정 방법을 컨텍스트에 주입
  → Claude 스스로 수정
  → 재검증 통과
```

핵심: 린터 에러 메시지에 **"어떻게 고쳐야 하는지"를 함께 포함**시키면, Claude가 스스로 수정하는 피드백 루프가 생깁니다.

#### ④ MCP 서버 — 토큰 비용을 고려해 연결

외부 서비스를 참조해야 할 때 MCP 서버로 연결합니다.  
단, MCP는 토큰을 많이 소비할 수 있으므로 **꼭 필요한 것만 연결**하세요.  
`/mcp` 명령으로 서버별 토큰 사용량을 주기적으로 확인하세요.

#### ⑤ 코드 가비지 컬렉션

AI가 코드를 대량 생성하다 보면 낡은 패턴이 복제되고 기술 부채가 쌓입니다.  
백그라운드 에이전트를 이용해 **주기적으로 편차를 스캔하고 리팩토링 PR을 자동 생성**하도록 설정할 수 있습니다.

### 팀 하네스 구축 체크리스트

```
□ CLAUDE.md 작성 — 목차형, 200줄 이하
□ .claude/rules/ 로 지침 모듈화
□ 린터 및 포맷터 설정 (ESLint, Prettier 등)
□ CI 파이프라인 구성 + Claude 피드백 루프 연결
□ MCP 서버 목록 정리 (토큰 비용 고려)
□ 세션 전환 기준 수립 (언제 /clear 할지 팀 공유)
□ 팀 공통 CLAUDE.md 템플릿 배포 및 버전 관리
```

---

## 빠른 참조 — 자주 쓰는 명령어

| 명령어 | 설명 |
|--------|------|
| `/init` | 프로젝트 분석 후 CLAUDE.md 초안 생성 |
| `/model` | 사용 모델 변경 (Opus / Sonnet) |
| `/context` | 컨텍스트 사용량 확인 |
| `/compact [지침]` | 대화 압축 |
| `/clear` | 대화 이력 초기화 |
| `/mcp` | MCP 서버 토큰 사용량 확인 |
| `/agents` | Subagent 구성 안내 |
| `/doctor` | 설치 문제 진단 |
| `Shift + Tab` | PlanMode 토글 |
| `Esc × 2` | 마지막 파일 편집 취소 |

---

> **핵심 요약**
>
> 1. **CLAUDE.md는 짧고 구체적으로** — Claude의 행동 기준이자 팀의 코딩 계약서
> 2. **컨텍스트를 자주 초기화하세요** — 품질 유지의 핵심. 단계마다 커밋하고 `/clear`
> 3. **큰 작업은 PlanMode로 먼저 계획** — 방향 정렬 후 실행
> 4. **실수가 생기면 CLAUDE.md에 기록** — 하네스는 사용할수록 단단해집니다
